In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

def build_graph(objects, label):
    x = torch.tensor(objects, dtype=torch.float)
    n = x.size(0)
    edge_index = torch.tensor(
        [[i, j] for i in range(n) for j in range(n) if i != j],
        dtype=torch.long
    ).t().contiguous()
    y = torch.tensor([label], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y)

class GNN(torch.nn.Module):
    
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch)
        x = self.lin(x)
        return x

events = [
    ([[50.0, 1.2, 0.3], [30.0, -0.5, 1.1]], 1),
    ([[20.0, 0.1, -1.0], [15.0, 2.0, 0.8], [10.0, -1.5, 2.2]], 0),
]

dataset = [build_graph(obj, lab) for obj, lab in events]
loader = DataLoader(dataset, batch_size=2, shuffle=True)

model = GNN(in_channels=3, hidden_channels=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

model.train()
for epoch in range(50):
    for batch in loader:
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = F.cross_entropy(out, batch.y)
        loss.backward()
        optimizer.step()

/tmp/ipykernel_68538/2005644622.py:38: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  loader = DataLoader(dataset, batch_size=2, shuffle=True)
